# Bronze Layer - pump.fun Event Ingestion

Near-real-time ingestion of pump.fun trading events into Delta.

The `ingestion/pumpfun` service (a standalone websocket client, see its
README) streams live events from the PumpAPI feed and lands them as JSONL
files in a Unity Catalog Volume. This notebook uses Auto Loader to pick up
new files incrementally and append them to `bronze.pumpfun_events`.

Each landed line has the shape `{"_source": ..., "_ingested_at": ..., "event": {...}}`.
Bronze keeps the full `event` payload as raw JSON text (no schema imposed) —
`event.action` varies (`buy`, `sell`, `create`, `migrate`, `createPool`,
`transfer`, ...) and each action has a different set of fields. See
`ingestion/pumpfun/docs/glossary_of_event_properties.txt` for field meanings.
Typed parsing happens in Silver (`processing/silver/NB_process_pumpfun_silver`).

In [ ]:
from pyspark.sql.functions import col, get_json_object, current_timestamp

dbutils.widgets.text("LANDING_PATH",     "/Volumes/workspace/default/mnt/pumpapi")
dbutils.widgets.text("CATALOG",          "workspace")
dbutils.widgets.text("BRONZE_SCHEMA",    "bronze")
dbutils.widgets.text("CHECKPOINT_PATH",  "/Volumes/workspace/default/mnt/checkpoints/bronze_pumpfun")
dbutils.widgets.text("SCHEMA_LOCATION",  "/Volumes/workspace/default/mnt/checkpoints/bronze_pumpfun_autoloader_schema")

CONFIG = {
    "landing_path":     dbutils.widgets.get("LANDING_PATH"),
    "catalog":          dbutils.widgets.get("CATALOG"),
    "bronze_schema":    dbutils.widgets.get("BRONZE_SCHEMA"),
    "checkpoint_path":  dbutils.widgets.get("CHECKPOINT_PATH"),
    "schema_location":  dbutils.widgets.get("SCHEMA_LOCATION"),
}

bronze_table_fqn = f"{CONFIG['catalog']}.{CONFIG['bronze_schema']}.pumpfun_events"
print(f"Target table: {bronze_table_fqn}")

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CONFIG['catalog']}.{CONFIG['bronze_schema']}")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {bronze_table_fqn} (
        source              STRING,
        ingested_at         TIMESTAMP,
        action              STRING,
        mint                STRING,
        signature           STRING,
        pool_id             STRING,
        event_json          STRING,
        raw_line            STRING,
        bronze_ingested_at  TIMESTAMP
    ) USING DELTA
""")
print(f"{bronze_table_fqn} is ready")

In [ ]:
# Auto Loader over the JSONL landing volume. cloudFiles.format is "text" —
# each file is JSON-lines, and we deliberately do NOT let Auto Loader infer a
# schema for the nested `event` object (it varies per action and would force
# a rigid/evolving schema onto a Bronze layer that should stay raw). Instead
# we keep the full line, plus a handful of get_json_object-derived columns
# for partition pruning / debugging, and pass the whole `event` object
# through as a JSON string for Silver to parse.
raw_lines = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format",         "text")
    .option("cloudFiles.schemaLocation", CONFIG["schema_location"])
    .load(CONFIG["landing_path"])
)

bronze_events = (
    raw_lines
    .withColumn("source",       get_json_object(col("value"), "$._source"))
    .withColumn("ingested_at",  get_json_object(col("value"), "$._ingested_at").cast("timestamp"))
    .withColumn("event_json",   get_json_object(col("value"), "$.event"))
    .withColumn("action",       get_json_object(col("event_json"), "$.action"))
    .withColumn("mint",         get_json_object(col("event_json"), "$.mint"))
    .withColumn("signature",    get_json_object(col("event_json"), "$.signature"))
    .withColumn("pool_id",      get_json_object(col("event_json"), "$.poolId"))
    .withColumn("bronze_ingested_at", current_timestamp())
    .withColumnRenamed("value", "raw_line")
    .select(
        "source", "ingested_at", "action", "mint", "signature", "pool_id",
        "event_json", "raw_line", "bronze_ingested_at",
    )
)

In [ ]:
def write_bronze(batch_df, batch_id):
    if not batch_df.take(1):
        return

    total = batch_df.count()
    malformed = batch_df.filter(col("event_json").isNull()).count()
    if malformed:
        print(f"  ⚠️ batch {batch_id}: {malformed}/{total} lines had no parseable 'event' field")

    (
        batch_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(bronze_table_fqn)
    )
    print(f"  batch {batch_id}: wrote {total} pump.fun events")

In [ ]:
query = (
    bronze_events.writeStream
    .foreachBatch(write_bronze)
    .option("checkpointLocation", CONFIG["checkpoint_path"])
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

print("✅ pump.fun Bronze ingestion completed")